In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from scipy import stats

from conus_biomass import dir_info
from conus_biomass.make_figures import maps
from conus_biomass.process_inputs import bin_nfi_plots
from conus_biomass.process_outputs.generate_state_csv import (
    clip_to_shape,
    get_crs,
    get_output_biomass,
)
from conus_biomass.train_models import train_all_models

In [ ]:
slope = xr.open_zarr(dir_info.dir_processed + "data_on_ref_grid/1000m/aspect.zarr")
crs = slope["spatial_ref"].crs_wkt

In [ ]:
fia_data = train_all_models.load_data(
    fpath=dir_info.dir_processed + "restructured_FIA/" + "*_FIA_plots_and_PRISM_v7.nc"
)
fia_data = fia_data.where(
    fia_data["STATECD"].isin([4, 6, 8, 16, 30, 32, 35, 41, 49, 53, 56]).load(), drop=True
)

In [ ]:
extracted_points = xr.open_dataset("modeled_biomass_all_years_fia_points_buffer.nc")

In [ ]:
extracted_points["plotid"] = extracted_points["plotid"].values.astype("U12")

In [ ]:
fia_data["biomass_modeled"] = extracted_points["predicted_biomass"].transpose("plotid", "year")

In [ ]:
# fia_data["biomass_modeled"]=fia_data["biomass_modeled"].where(~np.isnan(fia_data["biomass"]))

In [ ]:
from conus_biomass.train_models.train_models_utils import calculate_change_in_var

In [ ]:
modeled_delta = calculate_change_in_var(fia_data=fia_data, var="biomass_modeled").where(
    fia_data["measyear_2"] <= 2019
)

In [ ]:
actual_delta = calculate_change_in_var(fia_data=fia_data, var="biomass").where(
    fia_data["measyear_2"] <= 2019
)

In [ ]:
fia_data["biomass_delta_modeled"] = modeled_delta
fia_data["biomass_delta_actual"] = actual_delta

In [ ]:
nonanfilter = (~np.isnan(actual_delta).load()) * (~np.isnan(modeled_delta).load())
actual_delta_nonan = actual_delta[nonanfilter]
modeled_delta_nonan = modeled_delta[nonanfilter]
plt.hist2d(
    actual_delta_nonan, modeled_delta_nonan, bins=np.arange(-200, 100, 5), norm="log", vmax=1000
)
plt.colorbar()
plt.plot([-200, 160], [-200, 160], "-k")

In [ ]:
vars_year = ["biomass"]
vars_static = ["biomass_delta_actual", "lat", "lon"]

var_list = vars_year + vars_static
ds = fia_data[var_list]
ds_binned = bin_nfi_plots.calculate_ds_binned(ds=ds, vars_year=vars_year, vars_static=vars_static)
ds_binned_stacked, lon_2d, lat_2d = bin_nfi_plots.get_stacked_binned_data(
    plot_level_data=fia_data, vars_year=vars_year, vars_static=vars_static
)
ds_binned = ds_binned.rio.write_crs("EPSG:4326")
ds_binned["lat_bin"] = ds_binned["lat_bin"] + (0.5 / 2)
ds_binned["lon_bin"] = ds_binned["lon_bin"] + (0.5 / 2)

ds_binned_counts = bin_nfi_plots.calculate_ds_binned(
    ds=ds[["biomass_delta_actual", "lat", "lon"]],
    vars_year=vars_year,
    vars_static=vars_static,
    do_counts=True,
)
ds_binned_counts["lat_bin"] = ds_binned_counts["lat_bin"] + (0.5 / 2)
ds_binned_counts["lon_bin"] = ds_binned_counts["lon_bin"] + (0.5 / 2)

In [ ]:
vars_year = ["biomass"]
vars_static = ["biomass_delta_modeled", "lat", "lon"]

var_list = vars_year + vars_static
ds = fia_data[var_list]
ds_binned_modeled = bin_nfi_plots.calculate_ds_binned(
    ds=ds, vars_year=vars_year, vars_static=vars_static
)
ds_binned_modeld_stacked, lon_2d, lat_2d = bin_nfi_plots.get_stacked_binned_data(
    plot_level_data=fia_data, vars_year=vars_year, vars_static=vars_static
)
ds_binned_modeled = ds_binned_modeled.rio.write_crs("EPSG:4326")
ds_binned_modeled["lat_bin"] = ds_binned_modeled["lat_bin"] + (0.5 / 2)
ds_binned_modeled["lon_bin"] = ds_binned_modeled["lon_bin"] + (0.5 / 2)

ds_binned_modeled_counts = bin_nfi_plots.calculate_ds_binned(
    ds=ds[["biomass_delta_modeled", "lat", "lon"]],
    vars_year=vars_year,
    vars_static=vars_static,
    do_counts=True,
)
ds_binned_modeled_counts["lat_bin"] = ds_binned_modeled_counts["lat_bin"] + (0.5 / 2)
ds_binned_modeled_counts["lon_bin"] = ds_binned_modeled_counts["lon_bin"] + (0.5 / 2)

In [ ]:
inputs = xr.open_dataset(dir_info.dir_model_input + "all_variables.nc")
crs, grid_res = get_crs(dir_model_input=dir_info.dir_model_input)

FRF_filter = inputs["forest_remaining_forest"].sel(year=2015) > 0.99

da_end = get_output_biomass(year=2015, crs=crs)
da_start = get_output_biomass(year=2005, crs=crs)
delta_biomass = da_end - da_start
delta_biomass = delta_biomass.where(FRF_filter).rio.write_crs(crs)

delta_biomass_coarse = delta_biomass.coarsen(x=41, y=41, boundary="pad").mean().compute()
delta_biomass_western = clip_to_shape(delta_biomass_coarse, maps.SHP_WESTERN)
delta_biomass_western.load()

delta_biomass_coarse = delta_biomass.coarsen(x=4, y=4, boundary="pad").mean().compute()
delta_biomass_western_v2 = clip_to_shape(delta_biomass_coarse, maps.SHP_WESTERN)
delta_biomass_western_v2.load()

In [ ]:
target_crs = slope["spatial_ref"].crs_wkt

In [ ]:

fig, axs = plt.subplots(
    1, 4, figsize=(28, 9), gridspec_kw={"width_ratios": [0.05, 1, 1, 1]}, constrained_layout=True
)

font_settings = {
    "font.size": 28,
    "axes.titlesize": 30,
    "axes.labelsize": 28,
    "legend.fontsize": 28,
    "xtick.labelsize": 28,
    "ytick.labelsize": 28,
}
plt.rcParams.update(font_settings)
cmap = plt.cm.BrBG
clims = [-30, 30]
shp_native = maps.SHP_WESTERN.to_crs(target_crs)

counts_reproj = (
    ds_binned_counts["n_plots"]
    .rio.set_spatial_dims(x_dim="lon_bin", y_dim="lat_bin")
    .rio.write_crs("EPSG:4326")
    .rio.reproject(target_crs)
)
counts_modeled_reproj = (
    ds_binned_modeled_counts["n_plots"]
    .rio.set_spatial_dims(x_dim="lon_bin", y_dim="lat_bin")
    .rio.write_crs("EPSG:4326")
    .rio.reproject(target_crs)
)
biomass_delta_actual = (
    ds_binned["biomass_delta_actual"]
    .rio.set_spatial_dims(x_dim="lon_bin", y_dim="lat_bin")
    .rio.write_crs("EPSG:4326")
    .rio.clip_box(minx=-126, miny=31, maxx=-101, maxy=50)
    .rio.reproject(target_crs)
    .where(counts_reproj >= 10)
)
biomass_delta_modeled = (
    ds_binned_modeled["biomass_delta_modeled"]
    .rio.set_spatial_dims(x_dim="lon_bin", y_dim="lat_bin")
    .rio.write_crs("EPSG:4326")
    .rio.clip_box(minx=-126, miny=31, maxx=-101, maxy=50)
    .rio.reproject(target_crs)
    .where(counts_modeled_reproj >= 10)
)


def setup_ax(ax):
    ax.set_axis_off()
    shp_native.boundary.plot(ax=ax, color="gray", linewidth=1)


cbar_ax = axs[0]

ax = axs[1]
im = biomass_delta_actual.plot(ax=ax, vmin=clims[0], vmax=clims[1], cmap=cmap, add_colorbar=False)
ax.set_title("Mean Plot-level Observed ΔAGB")
setup_ax(ax)
ax.text(0.02, 0.98, "a", transform=ax.transAxes, fontsize=28 * 1.25, fontweight="bold", va="top")

ax = axs[2]
im = biomass_delta_modeled.plot(ax=ax, vmin=clims[0], vmax=clims[1], cmap=cmap, add_colorbar=False)
ax.set_title("Mean Plot-level Modeled ΔAGB")
setup_ax(ax)

cbar = fig.colorbar(im, cax=cbar_ax, orientation="vertical", extend="both")
cbar.set_label("ΔAGB between \n measurements (MgC/ha)")
cbar.ax.tick_params(labelsize=font_settings["xtick.labelsize"])
cbar_ax.yaxis.set_ticks_position("left")
cbar_ax.yaxis.set_label_position("left")
ax.text(0.02, 0.98, "b", transform=ax.transAxes, fontsize=28 * 1.25, fontweight="bold", va="top")

ax = axs[3]
x = ds_binned["biomass_delta_actual"].where(ds_binned_counts["n_plots"] >= 10).values.flatten()
y = (
    ds_binned_modeled["biomass_delta_modeled"]
    .where(ds_binned_modeled_counts["n_plots"] >= 10)
    .values.flatten()
)
nanfilter = (~np.isnan(x)) * (~np.isnan(y))

xmin = -75
xmax = 50
ax.plot(x[nanfilter], y[nanfilter], ".", markersize=15, alpha=0.5)
x_clean = x[nanfilter]
y_clean = y[nanfilter]

slope, intercept, r, p, se = stats.linregress(x_clean, y_clean)
r2 = r**2

x_fit = np.array([np.nanmin(x_clean), np.nanmax(y_clean)])
ax.plot(
    x_fit, slope * x_fit + intercept, "--", color="k", label=f"fit (y={slope:.2f}x+{intercept:.2f})"
)
ax.text(0.02, 0.85, f"y={slope:.2f}x + {intercept:.2f}", transform=ax.transAxes, fontsize=24)
ax.text(0.02, 0.78, f"R² = {r2:.2f}", transform=ax.transAxes, fontsize=24)
ax.set_xlabel("Mean plot-level observed \n ΔAGB (MgC/ha)")
ax.set_ylabel("Mean plot-level modeled \n ΔAGB (MgC/ha)")
ax.plot([xmin, xmax], [xmin, xmax], "k")
ax.axhline(y=0, linestyle=":", color="k")
ax.axvline(x=0, linestyle=":", color="k")
ax.set_xlim([xmin, xmax])
ax.set_ylim([xmin, xmax])
ticks = np.arange(xmin + 25, xmax + 1, 25)  # adjust step size as needed
ax.set_xticks(ticks)
ax.set_yticks(ticks)
ax.tick_params(axis="y", which="both", left=True, right=True, direction="in")
ax.tick_params(axis="x", which="both", bottom=True, top=True, direction="in")
ax.text(0.02, 0.98, "c", transform=ax.transAxes, fontsize=28 * 1.25, fontweight="bold", va="top")

fig.get_layout_engine().set(w_pad=0, h_pad=0, hspace=0, wspace=0.02)
fig.savefig(dir_info.dir_figures + "Figure4.jpg", bbox_inches="tight", dpi=300)